# **BIBLIOTECA 2.0** — Notebook final

Este notebook consolida o estado final do projeto, incluindo:
- contexto e objetivos;
- preparação e limpeza do dataset GoodReads (100k livros);
- funções de exploração (pesquisa, filtros, rankings);
- visualizações (ECharts: scatter e barras; e o visual combinado *Volume vs Rating*);
- integração com OpenStreetMap/Overpass para bibliotecas em Portugal;
- instruções de execução e notas de implementação.



## 1) Estrutura do projeto
- `GoodReads_100k_books.csv` — dataset original (fonte: *Kaggle*)
- `data.py` — carregamento, limpeza e tratamento dos dados
- `functions.py` — pesquisa, filtros, rankings, estatísticas e opções ECharts
- `app.py` — Menu interativo em Streamlit (UI), navegação e integração final


## 2) Requisitos e execução

### Dependências/Bibliotecas Python
- pandas
- pydeck
- requests
- streamlit
- streamlit-echarts

### Executar a app localmente
No terminal (na pasta do projeto):

```bash
streamlit run app.py
```

Se necessário, instala dependências:

```bash
pip install -r requirements.txt
```

## 3) Dataset

O ficheiro `GoodReads_100k_books.csv` contém informação de livros, como:
- `title`, `author`, `isbn`
- `rating`, `totalratings`
- `genre` 
- `description`, `img`, `link`


In [ ]:
import pandas as pd
import numpy as np

CSV_PATH = "GoodReads_100k_books.csv"  # ajustar se necessário


## 4) Módulo de dados (`data.py`)

A app usa um pequeno módulo de dados para:
- ler o CSV (`load_raw`);
- limpar e preparar o dataset (`clean`);
- guardar sugestões do utilizador (`save_suggestion`).

A limpeza do dataset trata:
- codificações/acentos em strings
- conversões numéricas (`rating`, `totalratings`, etc.)
- normalização de géneros (lista `genres_list`)
- remoção de valores problemáticos


In [ ]:
from pathlib import Path

def load_raw(path: str) -> pd.DataFrame | None:
    """Lê o CSV (devolve None se falhar)."""
    try:
        return pd.read_csv(path)
    except Exception:
        return None

def _safe_float_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def _safe_int_series(s: pd.Series) -> pd.Series:
    out = pd.to_numeric(s, errors="coerce")
    return out

def clean(df: pd.DataFrame) -> pd.DataFrame:
    """Limpeza leve e robusta para uso na app."""
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()

    # normalização de strings básicas
    for c in ["title", "author", "isbn", "desc", "link", "img", "genre"]:
        if c in out.columns:
            out[c] = out[c].astype(str).replace({"nan": ""}).fillna("").str.strip()

    # numéricos
    if "rating" in out.columns:
        out["rating"] = _safe_float_series(out["rating"])
    if "totalratings" in out.columns:
        out["totalratings"] = _safe_int_series(out["totalratings"])
    if "reviews" in out.columns:
        out["reviews"] = _safe_int_series(out["reviews"])
    if "ratings_count" in out.columns:
        out["ratings_count"] = _safe_int_series(out["ratings_count"])

    # genres_list: se existir 'genre' como string com separadores, cria lista
    if "genres_list" not in out.columns:
        if "genre" in out.columns:
            def _split_genres(x: str):
                if not x:
                    return []
                # separadores comuns
                parts = [p.strip() for p in str(x).replace(";", ",").split(",")]
                parts = [p for p in parts if p]
                return parts[:10]
            out["genres_list"] = out["genre"].apply(_split_genres)
        else:
            out["genres_list"] = [[] for _ in range(len(out))]

    # garantir book_id
    if "book_id" in out.columns:
        out["book_id"] = out["book_id"].astype(str)

    return out

def save_suggestion(title: str, author: str = "", genres: str = "", out_path: str = "suggestions.csv") -> None:
    """Guarda sugestões num CSV separado."""
    row = {
        "title": (title or "").strip(),
        "author": (author or "").strip(),
        "genres": (genres or "").strip(),
    }
    p = Path(out_path)
    if p.exists():
        base = pd.read_csv(p)
        base = pd.concat([base, pd.DataFrame([row])], ignore_index=True)
    else:
        base = pd.DataFrame([row])
    base.to_csv(p, index=False)


In [ ]:
df_raw = load_raw(CSV_PATH)
df = clean(df_raw) if df_raw is not None else pd.DataFrame()

df.shape, df.columns[:15]


## 5) Funções de exploração e visualização (`functions.py`)

A app organiza a lógica em funções:
- `search`: pesquisa por título/autor/ISBN
- `filter_by_genre`: filtro por género
- `most_popular`: ranking por nº de avaliações
- `hidden_gems`: rating alto com nº de avaliações moderado
- `quick_stats`: métricas e frases para a página “Sobre nós”
- `prep_genre_stats`: agregação por género (volume vs qualidade)
- `echarts_scatter_popularity_quality`: scatter popularidade vs qualidade
- `echarts_genre_bars`: duas barras lado-a-lado (volume e rating)
- `echarts_genre_volume_rating_combo`: visual combinado (barras + linha) para ver a relação volume vs rating



In [ ]:
def search(df: pd.DataFrame, query: str) -> pd.DataFrame:
    if df is None or df.empty:
        return df

    q = str(query).strip() if query is not None else ""
    if not q:
        return df

    q_lower = q.lower()

    author_col = "author" if "author" in df.columns else ("authors" if "authors" in df.columns else None)
    mask = pd.Series(False, index=df.index)

    if "title" in df.columns:
        mask |= df["title"].fillna("").astype(str).str.lower().str.contains(q_lower, na=False)

    if author_col:
        mask |= df[author_col].fillna("").astype(str).str.lower().str.contains(q_lower, na=False)

    res = df[mask].copy()
    if not res.empty:
        return res

    # fallback: ISBN (match exato)
    if "isbn" in df.columns:
        res = df[df["isbn"].fillna("").astype(str).str.strip() == q].copy()
        if not res.empty:
            return res

    return df.head(0)

def filter_by_genre(df: pd.DataFrame, genre: str) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    if not genre:
        return df

    g = str(genre).strip().lower()

    if "genres_list" in df.columns:
        mask = df["genres_list"].apply(
            lambda lst: isinstance(lst, (list, tuple)) and g in [str(x).lower() for x in lst]
        )
        return df[mask].copy()

    if "genre" in df.columns:
        mask = df["genre"].fillna("").astype(str).str.lower().str.contains(g, na=False)
        return df[mask].copy()

    return df.head(0)

def most_popular(df: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    if df is None or df.empty:
        return df

    col = None
    for candidate in ["totalratings", "reviews", "ratings_count"]:
        if candidate in df.columns:
            col = candidate
            break
    if col is None:
        return df.head(0)

    d = df.copy()
    d[col] = pd.to_numeric(d[col], errors="coerce").fillna(0)
    return d.sort_values(col, ascending=False).head(int(n))

def hidden_gems(df: pd.DataFrame, min_rating: float = 4.0, min_ratings: int = 50, max_ratings: int = 2000, n: int = 20) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    for col in ["rating", "totalratings"]:
        if col not in df.columns:
            return df.head(0)

    d = df.copy()
    d["rating"] = pd.to_numeric(d["rating"], errors="coerce")
    d["totalratings"] = pd.to_numeric(d["totalratings"], errors="coerce")

    d = d[
        (d["rating"] >= float(min_rating))
        & (d["totalratings"] >= int(min_ratings))
        & (d["totalratings"] <= int(max_ratings))
    ]
    if d.empty:
        return d

    d = d.sort_values(["rating", "totalratings"], ascending=[False, True])
    return d.head(int(n))

def quick_stats(df: pd.DataFrame) -> dict:
    if df is None or df.empty:
        return {"stats": {}, "lines": ["Sem dados."], "top_genres": []}

    stats = {
        "total_livros": int(len(df)),
        "total_autores": int(df["author"].dropna().nunique()) if "author" in df.columns else None,
    }

    ratings = pd.to_numeric(df["rating"], errors="coerce") if "rating" in df.columns else pd.Series([], dtype=float)
    stats["rating_medio"] = float(ratings.mean()) if ratings.notna().any() else None

    # top géneros (simple, via coluna 'genre' se existir)
    top_genres = []
    if "genre" in df.columns:
        g = df["genre"].dropna().astype(str).str.split(",", n=1).str[0].str.strip()
        bad = {"****", "", "nan", "none", "unknown"}
        g = g[g.str.lower().notna() & ~g.str.lower().isin(bad)]
        top_genres = list(g.value_counts().head(5).index)

    lines = []
    if stats["rating_medio"] is not None:
        lines.append(f"Rating médio do catálogo: {stats['rating_medio']:.2f}/5.")
    if top_genres:
        lines.append("Géneros mais frequentes: " + ", ".join(top_genres) + ".")
    return {"stats": stats, "lines": lines, "top_genres": top_genres}

def prep_genre_stats(df: pd.DataFrame, top_n: int = 12) -> pd.DataFrame:
    if df is None or df.empty or "genres_list" not in df.columns:
        return pd.DataFrame(columns=["genre", "n_books", "avg_rating", "median_rating", "avg_totalratings"])

    tmp = df[["genres_list", "rating", "totalratings"]].copy() if all(c in df.columns for c in ["rating","totalratings"]) else df[["genres_list","rating"]].copy()
    tmp["rating"] = pd.to_numeric(tmp.get("rating"), errors="coerce")
    if "totalratings" in tmp.columns:
        tmp["totalratings"] = pd.to_numeric(tmp.get("totalratings"), errors="coerce")

    tmp = tmp.explode("genres_list").rename(columns={"genres_list": "genre"})
    tmp["genre"] = tmp["genre"].astype(str).str.strip()
    tmp = tmp[tmp["genre"].ne("") & tmp["genre"].ne("nan")]

    agg = (
        tmp.groupby("genre", as_index=False)
        .agg(
            n_books=("genre", "size"),
            avg_rating=("rating", "mean"),
            median_rating=("rating", "median"),
            avg_totalratings=("totalratings", "mean") if "totalratings" in tmp.columns else ("rating", "size"),
        )
        .sort_values("n_books", ascending=False)
        .head(top_n)
    )
    return agg


## 6) Visualizações

A app usa `streamlit_echarts` para criar visuais interativos no Streamlit.  


### 6.1 Scatter: Popularidade vs Qualidade
- eixo X: nº de avaliações
- eixo Y: rating médio
- objetivo: perceber outliers e quadrantes (populares e bem avaliados; pérolas escondidas; etc.)

In [ ]:
def prep_popularity_vs_quality(df: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in ["title", "author", "genres_list", "rating", "totalratings"] if c in df.columns]
    if not cols:
        return pd.DataFrame(columns=["title", "author", "rating", "totalratings"])

    out = df[cols].copy()
    out["rating"] = pd.to_numeric(out.get("rating"), errors="coerce")
    out["totalratings"] = pd.to_numeric(out.get("totalratings"), errors="coerce")
    out = out.dropna(subset=["rating", "totalratings"])
    out = out[(out["totalratings"] > 0) & (out["rating"] > 0)]
    return out

def echarts_scatter_popularity_quality(df: pd.DataFrame) -> dict:
    d = prep_popularity_vs_quality(df)
    if d is None or d.empty:
        return {"series": []}

    if len(d) > 8000:
        d = d.sample(8000, random_state=42)

    y_ref = float(d["rating"].median())
    x_ref = float(d["totalratings"].quantile(0.75))

    source = []
    for _, r in d.iterrows():
        source.append({
            "votes": int(r["totalratings"]),
            "rating": float(r["rating"]),
            "title": str(r.get("title", "")).strip(),
            "author": str(r.get("author", "")).strip(),
        })

    return {
        "backgroundColor": "rgba(0,0,0,0)",
        "grid": {"left": 70, "right": 30, "top": 40, "bottom": 60},
        "dataset": {"source": source},
        "tooltip": {
            "trigger": "item",
            "confine": True,
            "backgroundColor": "rgba(20, 24, 32, 0.92)",
            "borderWidth": 0,
        },
        "xAxis": {
            "type": "log",
            "name": "Nº de Avaliações",
            "nameLocation": "middle",
            "nameGap": 30,
        },
        "yAxis": {
            "type": "value",
            "min": 1.5,
            "max": 5.5,
            "name": "Rating médio",
            "nameLocation": "middle",
            "nameRotate": 90,
            "nameGap": 30,
        },
        "series": [
            {
                "type": "scatter",
                "symbolSize": 7,
                "encode": {"x": "votes", "y": "rating", "tooltip": ["title", "author", "rating", "votes"]},
                "markLine": {
                    "silent": True,
                    "symbol": ["none", "none"],
                    "data": [
                        {"name": "Rating (mediana)", "yAxis": y_ref},
                        {"name": "Popularidade", "xAxis": x_ref},
                    ],
                },
            }
        ],
    }

scatter_opts = echarts_scatter_popularity_quality(df)
list(scatter_opts.keys()), len(scatter_opts.get("series", []))


### 6.2 Barras por género (duplo painel)

O `echarts_genre_bars` apresenta dois painéis de barros lado-a-lado:
- Nº de livros por género (volume)
- Rating médio por género (qualidade)


In [ ]:
def echarts_genre_bars(df: pd.DataFrame, top_n: int = 12) -> dict:
    gdf = prep_genre_stats(df, top_n=top_n)
    if gdf is None or gdf.empty:
        return {"series": []}

    # paleta retro/vintage (baixa saturação, sem brilho)
    palette = [
        "#3A3F45",  # charcoal
        "#6F7C6B",  # olive grey
        "#9FA68B",  # muted khaki
        "#C2BDA8",  # warm sand
        "#7F8F7A",  # moss green
        "#5E7A63",  # forest muted
        "#6E6A78",  # slate violet
        "#505B66",  # steel grey
        "#7A8796",  # blue-grey
        "#8F8176",  # taupe
        "#586B84",  # muted denim
        "#A08787",  # dusty rose
        "#4E647B",  # petrol blue
        "#6B7682",  # graphite
        "#9BA4AF",  # soft ash
        "#7A6A58",  # warm brown
    ]

    # mapa cor por género (consistente entre gráficos)
    # usa a ordem por n_books para fixar mapping de forma estável
    base_order = gdf.sort_values("n_books", ascending=False)["genre"].astype(str).tolist()
    color_map = {genre: palette[i % len(palette)] for i, genre in enumerate(base_order)}

    # duas ordenações diferentes, mas mesmas cores
    gdf_books = gdf.sort_values("n_books", ascending=True).copy()
    gdf_rating = gdf.sort_values("avg_rating", ascending=True).copy()

    genres_books = gdf_books["genre"].astype(str).tolist()
    values_books = [int(x) for x in gdf_books["n_books"].fillna(0).tolist()]
    bars_books = [{
            "value": v,
            "label": {"formatter": f"{v:,}".replace(",", "."),},
            "itemStyle": {"color": color_map.get(g, "rgba(30,41,59,0.55)")},}
        for g, v in zip(genres_books, values_books)
    ]

    genres_rating = gdf_rating["genre"].astype(str).tolist()
    values_rating = [round(float(x), 2) for x in gdf_rating["avg_rating"].fillna(0).tolist()]
    bars_rating = [
        {"value": v, "itemStyle": {"color": color_map.get(g, "rgba(30,41,59,0.55)")}}
        for g, v in zip(genres_rating, values_rating)
    ]

    # eixo rating “zoom”
    rmin = max(0, float(gdf["avg_rating"].min()) - 0.2)
    rmax = min(5, float(gdf["avg_rating"].max()) + 0.2)
    if rmax - rmin < 0.6:
        rmin = max(0, rmin - 0.2)
        rmax = min(5, rmax + 0.2)

    axis_common_hide_numbers = {
        "axisLabel": {"show": False},
        "axisTick": {"show": False},
        "axisLine": {"show": False},
        "splitLine": {"show": False},
    }

    option = {
        "backgroundColor": "rgba(0,0,0,0)",
        "grid": [
            {"left": 80, "right": 28, "top": 52, "bottom": 22, "width": "44%"},
            {"left": "66%", "right": 28, "top": 52, "bottom": 22, "width": "41%"},
        ],
        "title": [
            {"text": "Géneros por Número de Livros Publicados", "left": 12, "top": 10,
             "textStyle": {"fontSize": 13, "fontWeight": 600, "color": "#1f2430"}},
            {"text": "Géneros por Rating Médio", "left": "66%", "top": 10,
             "textStyle": {"fontSize": 13, "fontWeight": 600, "color": "#1f2430"}},
        ],
        "tooltip": {
            "trigger": "axis",
            "axisPointer": {"type": "shadow"},
            "confine": True,
            "backgroundColor": "rgba(20,24,32,.92)",
            "borderWidth": 0,
            "textStyle": {"color": "#fff", "fontFamily": "IBM Plex Mono, monospace"},
        },
        "xAxis": [
            {
                "type": "value",
                "gridIndex": 0,
                **axis_common_hide_numbers,
            },
            {
                "type": "value",
                "gridIndex": 1,
                "min": rmin,
                "max": rmax,
                **axis_common_hide_numbers,
            },
        ],
        "yAxis": [
            {
                "type": "category",
                "gridIndex": 0,
                "data": genres_books,
                "axisLabel": {"color": "#6b7280", "overflow": "truncate", "width": 150,
                              "fontFamily": "IBM Plex Mono, monospace"},
                "axisLine": {"show": False},
                "axisTick": {"show": False},
            },
            {
                "type": "category",
                "gridIndex": 1,
                "data": genres_rating,
                "axisLabel": {"color": "#6b7280", "overflow": "truncate", "width": 150,
                              "fontFamily": "IBM Plex Mono, monospace"},
                "axisLine": {"show": False},
                "axisTick": {"show": False},
            },
        ],
        "series": [
            {
                "name": "Nº de livros",
                "type": "bar",
                "xAxisIndex": 0,
                "yAxisIndex": 0,
                "data": bars_books,
                "barWidth": 14,
                "itemStyle": {"borderRadius": 6},
                "label": {
                    "show": True,
                    "position": "right",
                    "color": "#6b7280",
                    "fontFamily": "IBM Plex Mono, monospace",
                },
                "emphasis": {"focus": "self"},
                "blur": {"itemStyle": {"opacity": 0.15}},
            },
            {
                "name": "Rating médio",
                "type": "bar",
                "xAxisIndex": 1,
                "yAxisIndex": 1,
                "data": bars_rating,
                "barWidth": 14,
                "itemStyle": {"borderRadius": 6},
                "label": {
                    "show": True,
                    "position": "right",
                    "formatter": "{c}",
                    "color": "#6b7280",
                    "fontFamily": "IBM Plex Mono, monospace",
                },
                "emphasis": {"focus": "self"},
                "blur": {"itemStyle": {"opacity": 0.15}},
            },
        ],
    }
    return option


### 6.3 Visual combinado: Volume vs Rating por género

Para mostrar explicitamente a relação volume vs rating, foi criada uma função adicional (barras + linha):  `echarts_genre_volume_rating_combo`


In [ ]:
def echarts_genre_volume_rating_combo(df: pd.DataFrame, top_n: int = 12) -> dict:
    gdf = prep_genre_stats(df, top_n=top_n)
    if gdf is None or gdf.empty:
        return {"series": []}

    palette = [
        "#3A3F45", "#6F7C6B", "#9FA68B", "#C2BDA8",
        "#7F8F7A", "#5E7A63", "#6E6A78", "#505B66",
        "#7A8796", "#8F8176", "#586B84", "#A08787",
        "#4E647B", "#6B7682", "#9BA4AF", "#7A6A58",
    ]

    gdf = gdf.sort_values("n_books", ascending=False).copy()
    genres = gdf["genre"].astype(str).tolist()
    color_map = {genre: palette[i % len(palette)] for i, genre in enumerate(genres)}

    values_books = gdf["n_books"].astype(int).tolist()
    values_rating = gdf["avg_rating"].round(2).tolist()

    bars_books = [
        {"value": v, "itemStyle": {"color": color_map.get(g), "borderRadius": [6, 6, 0, 0]}}
        for g, v in zip(genres, values_books)
    ]

    rmin = max(0, float(min(values_rating)) - 0.2)
    rmax = min(5, float(max(values_rating)) + 0.2)

    return {
        "backgroundColor": "rgba(0,0,0,0)",
        "grid": {"left": 90, "right": 90, "top": 60, "bottom": 70},
        "title": {
            "text": "Volume vs Rating por Género",
            "left": "center",
            "top": 10,
            "textStyle": {"fontSize": 14, "fontWeight": 600, "color": "#1f2430"},
        },
        "legend": {
            "data": ["Rating médio"],  # apenas a linha
            "top": 32,
            "left": "center",
        },
        "tooltip": {
            "trigger": "axis",
            "axisPointer": {"type": "shadow"},
            "confine": True,
        },
        "xAxis": {
            "type": "category",
            "data": genres,
            "axisLabel": {"rotate": 35},
            "axisTick": {"show": False},
        },
        "yAxis": [
            {
                "type": "value",
                "name": "Nº de livros",
                "nameLocation": "middle",
                "nameRotate": 90,
                "nameGap": 60,
                "splitLine": {"lineStyle": {"color": "rgba(0,0,0,0.06)"}},
            },
            {
                "type": "value",
                "name": "Rating médio",
                "position": "right",
                "min": rmin,
                "max": rmax,
                "nameLocation": "middle",
                "nameRotate": 90,
                "nameGap": 60,
                "splitLine": {"show": False},
            },
        ],
        "series": [
            {"name": "Nº de livros", "type": "bar", "data": bars_books, "barWidth": 26},
            {"name": "Rating médio", "type": "line", "yAxisIndex": 1, "data": values_rating, "smooth": True, "symbol": "circle", "symbolSize": 7},
        ],
    }

combo_opts = echarts_genre_volume_rating_combo(df, top_n=12)
combo_opts.get("title", {}), len(combo_opts.get("series", []))


## 7) Integração: Bibliotecas Nacionais via OpenStreetMap (Overpass API)

A app consulta o Overpass API para obter as coordenadas/pontos de todas as bibliotecas em Portugal e mostrá-los num mapa interativo (PyDeck).


In [ ]:
import requests

def fetch_libraries_pt_osm() -> pd.DataFrame:
    query = """
    [out:json][timeout:60];
    area["name"="Portugal"]["boundary"="administrative"]["admin_level"="2"]->.pt;
    (
      nwr["amenity"="library"](area.pt);
    );
    out center tags;
    """
    url = "https://overpass-api.de/api/interpreter"
    r = requests.post(url, data={"data": query}, timeout=90)
    r.raise_for_status()
    data = r.json()

    rows = []
    for el in data.get("elements", []):
        tags = el.get("tags", {}) or {}
        lat = el.get("lat") or (el.get("center", {}) or {}).get("lat")
        lon = el.get("lon") or (el.get("center", {}) or {}).get("lon")
        if lat is None or lon is None:
            continue

        name = tags.get("name") or "Biblioteca (sem nome)"
        contact = tags.get("website") or tags.get("contact:website") or tags.get("phone") or tags.get("contact:phone") or ""
        rows.append({"name": str(name), "contact": str(contact), "lat": float(lat), "lon": float(lon), "source": "OSM"})

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out[out["lat"].between(36, 43) & out["lon"].between(-10.7, -5.0)]
    return out


## 8) App Streamlit (`app.py`)

O menu de interface do utilizador (UI) organiza-se por “páginas” via menu lateral:
- Home
- Explorar catálogo
- Explorar por género
- Livros mais populares
- Pérolas escondidas
- Insights & Tendências
- Sugerir novo livro
- Sobre nós


In [ ]:
if option == "Home":
    st.markdown(
    """
Descobre livros, autores e géneros através da plataforma literária mais popular do mundo.
Explora as melhores obras, as mais populares e encontra a tua próxima grande leitura!
"""
)
    
    # Espaço extra entre o texto introdutório e o menu do Home
    st.markdown("<div style='height: 90px;'></div>", unsafe_allow_html=True)
    
    c1, c2, c3 = st.columns(3, gap="medium")

    with c1:
        with st.container(border=True):
            st.subheader("Explorar catálogo")
            st.write("Pesquisa por título, autor ou ISBN.")
            if st.button("Abrir", use_container_width=True, key="go_catalog"):
                st.session_state.option = "Explorar catálogo"
                st.rerun()

    with c2:
        with st.container(border=True):
            st.subheader("Explorar por género")
            st.write("Encontra livros por género.")
            if st.button("Abrir", use_container_width=True, key="go_genre"):
                st.session_state.option = "Explorar por género"
                st.rerun()

    with c3:
        with st.container(border=True):
            st.subheader("Livros mais populares")
            st.write("Vê os mais avaliados e mais lidos.")
            if st.button("Abrir", use_container_width=True, key="go_popular"):
                st.session_state.option = "Livros mais populares"
                st.rerun()

    c4, c5, c6 = st.columns(3, gap="medium")

    with c4:
        with st.container(border=True):
            st.subheader("Pérolas escondidas")
            st.write("Boas leituras com menos hype.")
            if st.button("Abrir", use_container_width=True, key="go_gems"):
                st.session_state.option = "Pérolas escondidas"
                st.rerun()

    with c5:
        with st.container(border=True):
            st.subheader("Insights & Tendências")
            st.write("Explora as tendências do catálogo.")
            if st.button("Abrir", use_container_width=True, key="go_last"):
                st.session_state.option = "Insights & Tendências"
                st.rerun()

    with c6:
        with st.container(border=True):
            st.subheader("Estatísticas rápidas")
            st.write("Resumo do catálogo e curiosidades.")
            if st.button("Abrir", use_container_width=True, key="go_stats"):
                st.session_state.option = "Estatísticas rápidas"
                st.rerun()


elif option == "Explorar catálogo":
    st.subheader("Explorar catálogo")
    query = st.text_input("Texto a pesquisar (título, autor ou ISBN):")

    if query:
        res = search(df, query)

        if res is None or len(res) == 0:
            st.warning("Nenhum livro encontrado para essa pesquisa.")
        else:
            st.write(f"Foram encontrados {len(res)} livros.")
            view = make_view(res)

            st.markdown(
                "<small style='color:#888;'>Dica: usa Ctrl/Cmd para selecionar várias linhas, ou Shift para selecionar um intervalo.</small>",
                unsafe_allow_html=True,
            )

            view_ui = view.drop(columns=["_book_id"], errors="ignore")

            evt = st.dataframe(
                view_ui,
                use_container_width=True,
                hide_index=True,
                column_config=make_column_config(view_ui),
                selection_mode="multi-row",
                on_select="rerun",
            )

            rows = (evt.selection.rows if (evt and evt.selection) else [])
            n_sel = len(rows)

            if n_sel == 0:
                st.info("Seleciona um livro na tabela para veres os detalhes.")
            else:
                selected_df = view.iloc[rows].copy()
                scroll_to_details()

                # (NOVO) carrega bibliotecas 1 vez por rerun e reutiliza
                try:
                    df_lib_global = get_libraries_osm_cached()
                except Exception:
                    df_lib_global = pd.DataFrame()

                if n_sel == 1:
                    st.subheader("Detalhes do livro")
                    chosen_id = str(selected_df.iloc[0]["_book_id"])
                    book = get_book_detail_by_id(df, chosen_id)
                    if book:
                        render_book_detail(book, df_lib_global)
                    else:
                        st.info("Não foi possível obter os detalhes deste livro.")
                else:
                    st.subheader("Detalhes dos livros selecionados")

                    tabs_labels = []
                    for _, r in selected_df.iterrows():
                        title = str(r.get("title", "")).strip()
                        label = f"{title[:35]}{'…' if len(title) > 35 else ''}"
                        tabs_labels.append(label)

                    tabs = st.tabs(tabs_labels)

                    for tab, (_, r) in zip(tabs, selected_df.iterrows()):
                        with tab:
                            chosen_id = str(r["_book_id"])
                            book = get_book_detail_by_id(df, chosen_id)
                            if book:
                                render_book_detail(book, df_lib_global)
                            else:
                                st.info("Não foi possível obter os detalhes deste livro.")


elif option == "Explorar por género":
    st.subheader("Explorar por género")

    genres = (
        sorted(
            set(
                g
                for cell in df.get("genres_list", [])
                for g in (cell if isinstance(cell, list) else [])
            )
        )
        if "genres_list" in df.columns
        else []
    )

    genre = st.selectbox("Escolhe um género:", genres) if genres else None

    if not genres:
        st.info("Não foi possível inferir a lista de géneros a partir dos dados.")
    elif genre:
        res = filter_by_genre(df, genre)

        if res is None or len(res) == 0:
            st.warning(f"Nenhum livro encontrado para o género '{genre}'.")
        else:
            st.write(f"Foram encontrados {len(res)} livros no género '{genre}'.")
            view = make_view(res)

            st.markdown(
                "<small style='color:#888;'>Dica: usa Ctrl/Cmd para selecionar várias linhas, ou Shift para selecionar um intervalo.</small>",
                unsafe_allow_html=True,
            )

            view_ui = view.drop(columns=["_book_id"], errors="ignore")

            evt = st.dataframe(
                view_ui,
                use_container_width=True,
                hide_index=True,
                column_config=make_column_config(view_ui),
                selection_mode="multi-row",
                on_select="rerun",
            )

            rows = (evt.selection.rows if (evt and evt.selection) else [])
            n_sel = len(rows)

            if n_sel == 0:
                st.info("Seleciona um livro na tabela para veres os detalhes.")
            else:
                selected_df = view.iloc[rows].copy()
                scroll_to_details()

                # (NOVO) carrega bibliotecas 1 vez por rerun e reutiliza
                try:
                    df_lib_global = get_libraries_osm_cached()
                except Exception:
                    df_lib_global = pd.DataFrame()

                if n_sel == 1:
                    st.subheader("Detalhes do livro")
                    chosen_id = str(selected_df.iloc[0]["_book_id"])
                    book = get_book_detail_by_id(df, chosen_id)
                    if book:
                        render_book_detail(book, df_lib_global)
                    else:
                        st.info("Não foi possível obter os detalhes deste livro.")
                else:
                    st.subheader("Detalhes dos livros selecionados")

                    tabs_labels = []
                    for _, r in selected_df.iterrows():
                        bid = str(r["_book_id"])
                        title = str(r.get("title", "")).strip()
                        label = f"{bid} — {title[:35]}{'…' if len(title) > 35 else ''}"
                        tabs_labels.append(label)

                    tabs = st.tabs(tabs_labels)

                    for tab, (_, r) in zip(tabs, selected_df.iterrows()):
                        with tab:
                            chosen_id = str(r["_book_id"])
                            book = get_book_detail_by_id(df, chosen_id)
                            if book:
                                render_book_detail(book, df_lib_global)
                            else:
                                st.info("Não foi possível obter os detalhes deste livro.")


elif option == "Livros mais populares":
    st.subheader("Livros mais populares")
    n = st.slider("Número de livros a mostrar:", min_value=5, max_value=50, value=20)

    res = most_popular(df, n=n)
    if res is None or len(res) == 0:
        st.warning("Não foi possível obter a lista de livros mais populares.")
    else:
        view = make_view(res)

        st.markdown(
            "<small style='color:#888;'>Dica: usa Ctrl/Cmd para selecionar várias linhas, ou Shift para selecionar um intervalo.</small>",
            unsafe_allow_html=True,
        )

        view_ui = view.drop(columns=["_book_id"], errors="ignore")

        evt = st.dataframe(
            view_ui,
            use_container_width=True,
            hide_index=True,
            column_config=make_column_config(view_ui),
            selection_mode="multi-row",
            on_select="rerun",
        )

        rows = (evt.selection.rows if (evt and evt.selection) else [])
        n_sel = len(rows)

        if n_sel == 0:
            st.info("Seleciona um livro na tabela para veres os detalhes.")
        else:
            selected_df = view.iloc[rows].copy()
            scroll_to_details()

            # IMPORTANTE: carregar bibliotecas 1 vez e reutilizar
            try:
                df_lib_global = get_libraries_osm_cached()
            except Exception:
                df_lib_global = pd.DataFrame()

            # Debug opcional (remove depois)
            # st.caption(f"Bibliotecas carregadas: {len(df_lib_global)}")

            if n_sel == 1:
                st.subheader("Detalhes do livro")
                chosen_id = str(selected_df.iloc[0]["_book_id"])
                book = get_book_detail_by_id(df, chosen_id)
                if book:
                    render_book_detail(book, df_lib_global)  # <-- passa df_lib_global
                else:
                    st.info("Não foi possível obter os detalhes deste livro.")
            else:
                st.subheader("Detalhes dos livros selecionados")

                tabs_labels = []
                for _, r in selected_df.iterrows():
                    title = str(r.get("title", "")).strip()
                    label = f"{title[:35]}{'…' if len(title) > 35 else ''}"
                    tabs_labels.append(label)

                tabs = st.tabs(tabs_labels)

                for tab, (_, r) in zip(tabs, selected_df.iterrows()):
                    with tab:
                        chosen_id = str(r["_book_id"])
                        book = get_book_detail_by_id(df, chosen_id)
                        if book:
                            render_book_detail(book, df_lib_global)  # <-- e aqui também
                        else:
                            st.info("Não foi possível obter os detalhes deste livro.")


elif option == "Pérolas escondidas":
    st.subheader("Pérolas escondidas")
    min_rating = st.slider("Rating mínimo:", 0.0, 5.0, 4.0, 0.1)
    min_ratings = st.number_input("Mínimo de avaliações:", min_value=0, value=50)
    max_ratings = st.number_input("Máximo de avaliações:", min_value=1, value=2000)
    n = st.slider("Número máximo de livros:", min_value=5, max_value=50, value=20)

    res = hidden_gems(df, min_rating=min_rating, min_ratings=min_ratings, max_ratings=max_ratings, n=n)

    if res is None or len(res) == 0:
        st.warning("Não foram encontradas pérolas com estes critérios.")
    else:
        view = make_view(res)

        st.markdown(
            "<small style='color:#888;'>Dica: usa Ctrl/Cmd para selecionar várias linhas, ou Shift para selecionar um intervalo.</small>",
            unsafe_allow_html=True,
        )

        view_ui = view.drop(columns=["_book_id"], errors="ignore")

        evt = st.dataframe(
            view_ui,
            use_container_width=True,
            hide_index=True,
            column_config=make_column_config(view_ui),
            selection_mode="multi-row",
            on_select="rerun",
        )

        rows = (evt.selection.rows if (evt and evt.selection) else [])
        n_sel = len(rows)

        if n_sel == 0:
            st.info("Seleciona um livro na tabela para veres os detalhes.")
        else:
            selected_df = view.iloc[rows].copy()
            scroll_to_details()


            try:
                df_lib_global = get_libraries_osm_cached()
            except Exception:
                df_lib_global = pd.DataFrame()


            if n_sel == 1:
                st.subheader("Detalhes do livro")
                chosen_id = str(selected_df.iloc[0]["_book_id"])
                book = get_book_detail_by_id(df, chosen_id)
                if book:
                    render_book_detail(book, df_lib_global)  
                else:
                    st.info("Não foi possível obter os detalhes deste livro.")
            else:
                st.subheader("Detalhes dos livros selecionados")

                tabs_labels = []
                for _, r in selected_df.iterrows():
                    title = str(r.get("title", "")).strip()
                    label = f"{title[:35]}{'…' if len(title) > 35 else ''}"
                    tabs_labels.append(label)

                tabs = st.tabs(tabs_labels)

                for tab, (_, r) in zip(tabs, selected_df.iterrows()):
                    with tab:
                        chosen_id = str(r["_book_id"])
                        book = get_book_detail_by_id(df, chosen_id)
                        if book:
                            render_book_detail(book, df_lib_global)
                        else:
                            st.info("Não foi possível obter os detalhes deste livro.")


elif option == "Insights & Tendências":
    st.subheader("Insights & Tendências")
    st.write("Uma leitura rápida do catálogo: popularidade, qualidade e padrões por género.")

    st.markdown("<div style='height: 50px;'></div>", unsafe_allow_html=True)

    # =========================
    # 1) Géneros: quantidade vs qualidade
    # =========================
    st.markdown("### Quantidade vs Qualidade")
    st.caption("Alguns géneros dominam em volume, mas outros destacam-se na avaliação média.")

    use_relation = st.toggle("Mostrar relação Volume vs Rating", value=True)

    if use_relation:
        option_genres = echarts_genre_volume_rating_combo(df, top_n=12)
        height_genres = "520px"
    else:
        option_genres = echarts_genre_bars(df, top_n=12)
        height_genres = "620px"

    if not option_genres.get("series"):
        st.info("Não há dados de géneros suficientes para esta análise.")
    else:
        st_echarts(
            options=option_genres,
            height=height_genres,
        )

        # Storytelling final (mantém)
        gdf = prep_genre_stats(df, top_n=12)
        if gdf is not None and not gdf.empty:
            best = gdf.sort_values("avg_rating", ascending=False).iloc[0]
            st.caption(
                f"Entre os géneros mais representados, "
                f"'{best['genre']}' tem o maior rating médio ({best['avg_rating']:.2f})."
            )

    st.markdown("<div style='height: 20px;'></div>", unsafe_allow_html=True)
    st.markdown("---")

    # =========================
    # 2) Popularidade vs Qualidade
    # =========================
    st.markdown("### Popularidade vs Qualidade")
    st.caption("Nem sempre os livros mais populares são os mais bem avaliados.")

    option_scatter = echarts_scatter_popularity_quality(df)

    if not option_scatter.get("series"):
        st.info("Não há dados suficientes para gerar este gráfico.")
    else:
        st_echarts(
            options=option_scatter,
            height="480px",
        )

elif option == "Sugerir novo livro":
    st.subheader("Sugerir um novo livro")
    title = st.text_input("Título do livro:")
    author = st.text_input("Autor(a):")
    genres = st.text_input("Género(s) (opcional):")

    if st.button("Submeter sugestão"):
        if not title:
            st.warning("É necessário indicar pelo menos o título.")
        else:
            save_suggestion(title, author, genres)
            st.success("Obrigado pela tua sugestão!")


elif option == "Sobre nós":

    st.markdown("<div style='height: 30px;'></div>", unsafe_allow_html=True)

    out = quick_stats(df)
    stats = out.get("stats", {})
    c1, c2, c3 = st.columns(3)

    def format_br(n):
        if n is None:
            return "—"
        return f"{n:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")

    with c1:
        st.metric("Livros", format_br(stats.get("total_livros")))
    with c2:
        st.metric("Autores", format_br(stats.get("total_autores")))
    with c3:
        rm = stats.get("rating_medio", None)
        st.metric("Rating médio", f"{float(rm):.2f}".rstrip("0").rstrip(".") if isinstance(rm, (int, float)) else "—")

    for line in out.get("lines", []):
        st.write(line)

